# obr04.ru — каталог организаций образования РА

Источник: портал Минобрнауки РА <https://obr04.ru/organization/> (обход всех 14 страниц каталога,
стр. 15+ повторяют первую — пагинация Bitrix зацикливается). Собрано 131 организация с их сайтами.
Тип выводится из названия (стандартные формулировки). Район — из названия по насёленным пунктам
из `data/dataset_v3/settlements.csv`.

Оговорка: портал перечисляет организации, имеющие сайты на нём, — это не полный реестр
(полные счётчики школ/садов — в ФРМО и ГИС ЖКХ).

In [1]:
import json, re
import pandas as pd
from pathlib import Path

REPO = Path('/home/mfthesun/Projects/altai-spatial-planning')
raw = json.loads((REPO / 'data/education/obr04_raw.json').read_text())
print('организаций:', len(raw), 'уникальных url:', len({r["url"] for r in raw}))
raw[:3]

организаций: 131 уникальных url: 131


[{'name': 'Муниципальное бюджетное общеобразовательное учреждение "Средняя общеобразовательная школа №7 г.Горно-Алтайска"',
  'url': 'https://7.obr04.ru/'},
 {'name': 'Бюджетное общеобразовательное учреждение Республики Алтай «Республиканская гимназия имени В.К. Плакаса»',
  'url': 'https://rg.obr04.ru/'},
 {'name': 'Бюджетное общеобразовательное учреждение Республики Алтай «Республиканский классический лицей»',
  'url': 'https://rcl.obr04.ru/'}]

In [4]:
by_type = json.loads((REPO / 'data/education/obr04_by_type.json').read_text())
rows = []
for typ, items in [('школа', by_type['school']), ('детский сад', by_type['detsad']), ('ДОД', by_type['dou'])]:
    for it in items:
        rows.append({'название': it['name'], 'сайт': it['url'], 'тип': typ})
df = pd.DataFrame(rows).drop_duplicates(subset=['название', 'сайт']).reset_index(drop=True)
print('всего организаций:', len(df))
print(df['тип'].value_counts())

# район: по насёленному пункту из названия (settlements.csv)
setts = pd.read_csv(REPO / 'data/dataset_v3/settlements.csv')
print('колонки settlements:', setts.columns.tolist()[:8])

всего организаций: 179
тип
школа          131
детский сад     32
ДОД             16
Name: count, dtype: int64
колонки settlements: ['object_name;settlement;mun_upper;mun_lower;oktmo;population;latitude;longitude;Указавшие национальную принадлежность;Русские (русаки', ' сибиряки);Алтайцы;Казахи;Теленгиты;Тубалары;Челканцы;Кумандинцы;Шорцы;Украинцы;Немцы (русские немцы);Киргизы (кыргыз);oktmo_mo;mo;Доля: Алтайцы', ' %;Доля: Русские', ' %;Доля: Казахи', ' %;Доля: Теленгиты', ' %;Доля: Тубалары', ' %;Доля: Кумандинцы', ' %;Доля: Челканцы']


In [5]:
setts = pd.read_csv(REPO / 'data/dataset_v3/settlements.csv', sep=';')
print(len(setts), 'НП; колонки:', setts.columns.tolist()[:8])
print(setts[['settlement', 'mo']].head(3).to_string())

247 НП; колонки: ['object_name', 'settlement', 'mun_upper', 'mun_lower', 'oktmo', 'population', 'latitude', 'longitude']
         settlement                  mo
0  г. Горно-Алтайск    г. Горно-Алтайск
1        село Майма    Майминский район
2     село Кош-Агач  Кош-Агачский район


In [6]:
# сопоставление: ищем название НП внутри названия организации (без «село/с.»)
def norm(s):
    return s.lower().replace('ё', 'е')

sett_list = []
for _, r in setts.iterrows():
    nm = re.sub(r'^(село|с |с\.\s*|поселок|п |п\.\s*|город|г )\s*', '', str(r['settlement']), flags=re.I)
    sett_list.append((norm(nm).strip(), r['settlement'], r['mo'], r['latitude'], r['longitude']))
sett_list.sort(key=lambda x: -len(x[0]))

def match_org(name):
    n = norm(name)
    for short, full, mo, lat, lon in sett_list:
        if len(short) >= 4 and short in n:
            return pd.Series([full, mo, lat, lon])
    return pd.Series([None, None, None, None])

df[['НП', 'район', 'lat', 'lon']] = df['название'].apply(match_org)
print('определён район:', df['район'].notna().sum(), 'из', len(df))
print('\nбез района:'); print(df.loc[df['район'].isna(), 'название'].tolist())

определён район: 87 из 179

без района:
['Муниципальное бюджетное общеобразовательное учреждение "Средняя общеобразовательная школа №7 г.Горно-Алтайска"', 'Бюджетное общеобразовательное учреждение Республики Алтай «Республиканская гимназия имени В.К. Плакаса»', 'Бюджетное общеобразовательное учреждение Республики Алтай «Республиканский классический лицей»', 'Казённое общеобразовательное учреждение Республики Алтай "Вечерняя (сменная) общеобразовательная школа"', 'КОУ РА "Коррекционная школа-интернат"', 'Казенное общеобразовательное учреждение Республики Алтай "Специальная (коррекционная) общеобразовательная школа-интернат для детей с нарушением слуха"', 'Казенное общеобразовательное учреждение Республики Алтай «Школа им. Г.К. Жукова»', 'МБОУ "Гимназия № 9 "Гармония" г.Горно-Алтайска"', 'МБОУ "Тихоньская основная общеобразовательная школа" им. Героя Советского Союза Т.И.Паршуткина', 'Муниципальное бюджетное общеобразовательное учреждение "Усть-Коксинская средняя общеобразовательная школ

In [13]:
# второй проход: прилагательные от НП («Талдинская» ← Талда, «Усть-Коксинская» ← Усть-Кокса)

def norm2(s):
    return norm(s).replace(' - ', '-').replace('—', '-')

city = next(x for x in sett_list if 'горно-алтайск' in x[0])

def adj_to_sett(stem):
    """стем прилагательного -> НП: сравнение с основой названия НП"""
    best = None
    for short, full, mo, lat, lon in sett_list:
        b = short[:-1] if short.endswith('а') or short.endswith('я') else short
        if stem.startswith(b) or b.startswith(stem):
            overlap = min(len(stem), len(b))
            if overlap >= 3 and (best is None or overlap > best[0]):
                best = (overlap, full, mo, lat, lon)
    return best[1:] if best else None

def match2(name, cur):
    if isinstance(cur[1], str):
        return cur
    n = norm2(name)
    if 'горно-алтайск' in n or 'республиканск' in n or n.startswith('коу ра') or 'ау до ра' in n or 'бпоу ра' in n:
        return (city[1], city[2], city[3], city[4])
    for w in re.findall(r'[а-яё-]+ская', n):
        hit = adj_to_sett(w[:-4])
        if hit:
            return hit
    for stm in ['Майминский', 'Усть-Коксинский', 'Шебалинский', 'Онгудайский', 'Кош-Агачский',
                'Турочакский', 'Улаганский', 'Усть-Канский', 'Чемальский', 'Чойский']:
        if stm.lower().replace('ё', 'е') in n:
            mo = setts.loc[setts['mo'].str.startswith(stm.split()[0]), 'mo']
            if len(mo):
                return (None, mo.iloc[0], None, None)
    return (None, None, None, None)

COLS = ['НП', 'район', 'lat', 'lon']
df2 = df.copy()
vals = [match2(r['название'], (r['НП'], r['район'], r['lat'], r['lon'])) for _, r in df.iterrows()]
df2[COLS] = pd.DataFrame(vals, index=df2.index, columns=COLS)
print('определён район:', df2['район'].notna().sum(), 'из', len(df2))
print('\nвсё ещё без района:'); print(df2.loc[df2['район'].isna(), 'название'].tolist())

определён район: 150 из 179

всё ещё без района:
['Казённое общеобразовательное учреждение Республики Алтай "Вечерняя (сменная) общеобразовательная школа"', 'Казенное общеобразовательное учреждение Республики Алтай "Специальная (коррекционная) общеобразовательная школа-интернат для детей с нарушением слуха"', 'Казенное общеобразовательное учреждение Республики Алтай «Школа им. Г.К. Жукова»', 'Муниципальное бюджетное общеобразовательное учреждение "Банновская основная общеобразовательная школа"', 'Муниципальное бюджетное общеобразовательное учреждение "Бело-Ануйская средняя общеобразовательная школа им. А.Х. Вязникова"', 'Муниципальное бюджетное общеобразовательное учреждение «Дьектиекская средняя общеобразовательная школа»', 'Муниципальное казенное общеобразовательное учреждение «Ининская средняя общеобразовательная школа»', 'Муниципальное бюджетное общеобразовательное учреждение "Мало-Чергинская основная общеобразовательная школа"', 'Муниципальное казенное общеобразовательное учрежден

In [14]:
# итог: CSV + сводка по районам
df2['район'] = df2['район'].fillna('не определён')
EDU = REPO / 'data/education'
df2[['название', 'тип', 'район', 'НП', 'сайт', 'lat', 'lon']].to_csv(EDU / 'obr04_organizations.csv',
                                                                    index=False, encoding='utf-8-sig')
print('сохранено:', EDU / 'obr04_organizations.csv', len(df2), 'строк')

sv = df2.pivot_table(index='район', columns='тип', values='название', aggfunc='count', fill_value=0)
sv['итого'] = sv.sum(axis=1)
print(sv.to_string())
sv.to_csv(EDU / 'obr04_summary.csv', encoding='utf-8-sig')

сохранено: /home/mfthesun/Projects/altai-spatial-planning/data/education/obr04_organizations.csv 179 строк
тип                    ДОД  детский сад  школа  итого
район                                                
Кош-Агачский район       0            0     12     12
Майминский район         1           13     12     26
Онгудайский район        2            0      9     11
Турочакский район        2            2      5      9
Улаганский район         1            0      9     10
Усть-Канский район       0            3     13     16
Усть-Коксинский район    3            0     18     21
Чемальский район         1            0      9     10
Чойский район            0            0      6      6
Шебалинский район        0            1     10     11
г. Горно-Алтайск         4            6      8     18
не определён             2            7     20     29
